# SHAP Dependence & Interaction Plots - all 8 years
Answers Reviewer #1 Q4 (dependence/interaction plots in addition to summary plots)
and Reviewer #2 Q5 (NDBI x SAVI and NDBI x NDWI interaction plots).

For each year (1990-2025) this produces:
- 4 dependence plots - one per predictor (NDVI, NDBI, NDWI, SAVI), each coloured by the strongest interacting feature (auto-detected by SHAP).
- 2 key interaction plots - NDBI coloured by SAVI, and NDBI coloured by NDWI.

Each RF uses the optimized hyperparameters from Table R4 (per year).

Setup: put this notebook in the folder with 1990.csv ... 2025.csv, then run top to bottom.
Requires: pip install shap scikit-learn pandas numpy matplotlib. Figures save to shap_figures/ at 300 dpi.

## 1. Settings

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Times New Roman"

HERE = ""                       # "" = same folder as this notebook
OUTDIR = os.path.join(HERE, "shap_figures")
os.makedirs(OUTDIR, exist_ok=True)

YEARS  = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025]
PREDS  = ["NDVI", "NDBI", "NDWI", "SAVI"]
SAMPLE = 2000                   # pixels sampled for SHAP (matches your original script)
SEED   = 42

# Table R4 - optimized RF hyperparameters per year
BEST = {
    1990: dict(n_estimators=500, max_depth=10, min_samples_split=5, min_samples_leaf=1, max_features="sqrt"),
    1995: dict(n_estimators=500, max_depth=10, min_samples_split=2, min_samples_leaf=2, max_features="sqrt"),
    2000: dict(n_estimators=200, max_depth=10, min_samples_split=2, min_samples_leaf=2, max_features="sqrt"),
    2005: dict(n_estimators=500, max_depth=10, min_samples_split=2, min_samples_leaf=2, max_features="sqrt"),
    2010: dict(n_estimators=200, max_depth=20, min_samples_split=5, min_samples_leaf=2, max_features="sqrt"),
    2015: dict(n_estimators=200, max_depth=10, min_samples_split=5, min_samples_leaf=2, max_features="sqrt"),
    2020: dict(n_estimators=500, max_depth=10, min_samples_split=2, min_samples_leaf=1, max_features="sqrt"),
    2025: dict(n_estimators=500, max_depth=10, min_samples_split=2, min_samples_leaf=1, max_features="sqrt"),
}
print("Ready. Figures ->", os.path.abspath(OUTDIR))

## 2. Helpers

In [ ]:
def load_year(y):
    df = pd.read_csv(os.path.join(HERE, f"{y}.csv"))
    cols = {}
    for p in PREDS + ["LST"]:
        hit = [c for c in df.columns if c.upper() == f"{p}_{y}".upper()]
        cols[p] = hit[0]
    keep = [cols[p] for p in PREDS] + [cols["LST"]]
    d = df[keep].apply(pd.to_numeric, errors="coerce").replace([-9999, -9999.0], np.nan).dropna()
    X = d[[cols[p] for p in PREDS]].copy()
    X.columns = PREDS                      # clean names (strip _YEAR) for nice axis labels
    y_ = d[cols["LST"]].values
    return X, y_

def fit_rf(X, y, year):
    rf = RandomForestRegressor(random_state=SEED, n_jobs=-1, **BEST[year])
    rf.fit(X, y)
    return rf

def shap_for(X, rf):
    Xs = X.sample(n=min(SAMPLE, len(X)), random_state=SEED)
    expl = shap.TreeExplainer(rf)
    sv = expl.shap_values(Xs)
    return Xs, sv

print("helpers defined")

## 3. Generate all plots
Runs all 8 years. Each figure is shown inline and saved to shap_figures/.

In [ ]:
for y in YEARS:
    print(f"\n===== {y} =====", flush=True)
    X, yv = load_year(y)
    rf = fit_rf(X, yv, y)
    Xs, sv = shap_for(X, rf)

    # ---- (a) dependence plot for each predictor (auto interaction colour) ----
    for p in PREDS:
        plt.figure(figsize=(6, 4.5))
        shap.dependence_plot(p, sv, Xs, interaction_index="auto", show=False)
        plt.title(f"{y} SHAP dependence: {p}", fontsize=13)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTDIR, f"dep_{y}_{p}.png"), dpi=300, bbox_inches="tight")
        plt.show()

    # ---- (b) KEY interaction plots requested by Reviewer #2 Q5 ----
    for main, inter in [("NDBI", "SAVI"), ("NDBI", "NDWI")]:
        plt.figure(figsize=(6, 4.5))
        shap.dependence_plot(main, sv, Xs, interaction_index=inter, show=False)
        plt.title(f"{y} SHAP interaction: {main} x {inter}", fontsize=13)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTDIR, f"interaction_{y}_{main}_x_{inter}.png"),
                    dpi=300, bbox_inches="tight")
        plt.show()

print("\nAll dependence and interaction plots saved to", os.path.abspath(OUTDIR))

## 4. (Optional) SHAP interaction strength
For each year, prints the mean absolute SHAP interaction value between NDBI and each of
SAVI / NDWI. Samples 800 pixels (slower). Saves SHAP_interaction_strength.csv.

In [ ]:
rows = []
for y in YEARS:
    X, yv = load_year(y)
    rf = fit_rf(X, yv, y)
    Xs = X.sample(n=min(800, len(X)), random_state=SEED)
    expl = shap.TreeExplainer(rf)
    inter = expl.shap_interaction_values(Xs)   # (n, 4, 4)
    idx = {p: i for i, p in enumerate(PREDS)}
    m = np.abs(inter).mean(axis=0)             # mean |interaction|
    rows.append({"Year": y,
                 "NDBIxSAVI": round(m[idx["NDBI"], idx["SAVI"]], 4),
                 "NDBIxNDWI": round(m[idx["NDBI"], idx["NDWI"]], 4)})
    print(f"{y}: NDBIxSAVI={rows[-1]['NDBIxSAVI']:.4f}  NDBIxNDWI={rows[-1]['NDBIxNDWI']:.4f}", flush=True)

pd.DataFrame(rows).to_csv(os.path.join(HERE, "SHAP_interaction_strength.csv"), index=False)
print("\nSaved SHAP_interaction_strength.csv")